# Batch Cellpose inference over a folder

- In this exercise, we will try to call Cellpose automatically on each image from a folder.
- At the end, we simply want a CSV containing the number of cells for each image.
- The first thing that we will need to do is to import all the modules that we are going to use.
    - `tifffile`: Useful to read and write TIFF files.
    - `pathlib.Path`: Allows to manipulate file/folder paths.
    - `numpy`: To manipulate N-dimensional arrays of data (like images).
    - `pandas`: Implements a DataFrame object allowing to manipulate CSV-shaped data.
    - `cellpose_napari`: Implements headless workers for common tasks realized with Cellpose. They are exposed through Napari's interface but can be called from anywhere.

In [ ]:
import tifffile as tiff
from pathlib import Path
import numpy as np
import pandas as pd
from cellpose_napari import CellPoseBatchWorker

- At this point we will store in variables: 
    - the location of our images 
    - where we would like the results to be saved
- To avoid errors by reading non-image files, we will specify the extension of what we are interested in. In our case we have **tif** files.

In [ ]:
images_folder_path = Path("/media/clement/3b801c96-393a-4b2e-be1e-9cabfbb10740/data-formation-ml-dl/S-BIAD1089/BIA/scaled")
labels_folder_path = Path("/media/clement/3b801c96-393a-4b2e-be1e-9cabfbb10740/data-formation-ml-dl/S-BIAD1089/BIA/segmented")
images_extension = "tif"

- To make sure that we caught everything, we will display in an array a preview of the data that we found.
- The first column "Source" is the name, then each column "Axis i" is the length of a dimension of the image.
- There should be as many axes as there are dimensions (ex: 2 for a 2D mono-channel image, 5 for 3D+t multi-channel image, ...)

In [ ]:
images_list = list(images_folder_path.glob(f"*.{images_extension}"))
print(f"{len(images_list)} images found in {images_folder_path} with extension '{images_extension}'.")

data = []
for image_path in images_list:
    image = tiff.imread(image_path)
    row = {'Source': image_path.name}
    for axis, size in enumerate(image.shape):
        row[f'Axis {axis}'] = size
    data.append(row)

images_df = pd.DataFrame(data)
display(images_df)

- At this point, we have to provide the common settings required by Cellpose.

| Variable | Description |
|:---|:---|
| `main_channel_prefix` | Prefix used to identify the main input channel files for segmentation. |
| `sec_channel_prefix` | Optional prefix for a second channel (set `None` if not used). |
| `axes_order` | Axis order expected in the images. |
| `pixel_size_yx` | Physical pixel size in Y/X (used for scale-aware processing). |
| `pixel_size_z` | Physical Z spacing for 3D data (unused for 2D images). |
| `model` | Cellpose model variant selected for inference (model name or absolute path). |
| `median_diameter` | Expected median object diameter in pixels. |
| `min_obj_size` | Minimum object size (in pixels) kept after segmentation. |
| `use_gpu` | Whether to run inference on GPU when available. |
| `cell_prob_threshold` | Cell probability threshold controlling mask acceptance. |
| `flow_threshold` | Flow consistency threshold for mask validation. |
| `flow_smoothing` | Flow-field smoothing factor before mask computation. |
| `segmentation_prefix` | Prefix added to output segmentation file names. |

In [ ]:
# Common settings
main_channel_prefix = "edl-"
sec_channel_prefix  = None
axes_order          = "YX"
pixel_size_yx       = 0.245 # in physical units
pixel_size_z        = None
model               = 'cpsam'
median_diameter     = 120 # in pixels
min_obj_size        = 20

# Extra settings
use_gpu             = True
cell_prob_threshold = 0.0
flow_threshold      = 0.4
flow_smoothing      = 1
segmentation_prefix = "seg-edl-"

- The last step is to instanciate Cellpose's batch worker and to feed it all these settings.
- The `run` method returns a generator, its result must be consumed for the code to be executed (lazy execution). It simply means that you can't just call run() alone, you have to do something (or pretend to do something) with its result.
- The `getLastItem` method of the worker returns a pair formed as follows: `(image_name: str, data: xr.dataArray)`.
- It means that if you wish, you can do something for each image on the fly (for example, display the number of cells found here.)
- We can even add items to a dataframe as we go to output a summary right away.

In [ ]:
df = pd.DataFrame(columns=['Image', 'Num. cells']) # Empty DataFrame with 'Image' and 'Num. cells' columns to store the summary.

wpbw = CellPoseBatchWorker(
    images_folder_path, 
    labels_folder_path, 
    main_channel_prefix, 
    sec_channel_prefix, 
    axes_order, 
    pixel_size_yx, 
    pixel_size_z, 
    model, 
    median_diameter, 
    min_obj_size, 
    use_gpu, 
    cell_prob_threshold, 
    flow_threshold, 
    flow_smoothing, 
    segmentation_prefix
)

for t in wpbw.run():
    image_name, data = wpbw.getLastItem()
    nCells = set(np.unique(data.values)) - {0} # 0 == background
    row = {'Image': image_name, 'Num. cells': len(nCells)} # Create a new row
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)

display(df)
df.to_csv(labels_folder_path / "segmentation_summary.csv", index=False) # Save on the disk.